In [8]:
"""
침수 취약도(V) = 노출(E, F) × 민감도(S) / 적응능력(C) 공식을 이용해
취약도 등급 1~5 를 예측하는 XGBoost 분류 모델 학습/최적화

- F(노출): FLOOD_AREA, RAIN_TOTAL, FLOOD_MONTH
- S(민감도): RIV_GRD, RIV_DIS_GRD, RIV_DIS_MIN
- C(적응능력): DRAINAGE_GRD, PUMP_CNT

V = (F * S) / C 로 연속 취약도 지수를 만든 뒤,
V를 5구간(quintile)으로 나누어 1~5 등급 레이블로 변환하여 분류 타깃으로 사용합니다.
"""

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score

# 1. 데이터 로드 -----------------------------------------------------------------
DATA_PATH = "../data/processed/total_data.csv"

# CSV 파일 읽기 (문자/숫자 혼합 컬럼 경고를 피하기 위해 dtype 지정/변환)
df = pd.read_csv(DATA_PATH, low_memory=False)

# 사용할 주요 숫자 컬럼 목록 -----------------------------------------------------
num_cols = [
    "FLOOD_AREA",
    "RAIN_TOTAL",
    "FLOOD_MONTH",      # 침수 발생 월 (첫 번째 FLOOD_MONTH)
    "DRAINAGE_GRD",
    "PUMP_CNT",
    "RIV_GRD",
    "RIV_DIS_MIN",
    "RIV_DIS_GRD",
]

# 안전하게 숫자로 변환(예: '-' 등을 NaN 으로) 후, 결측치 행 제거
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    else:
        raise ValueError(f"필요한 컬럼 '{c}' 이(가) 데이터에 없습니다. df.columns 를 확인하세요.")

before_drop = len(df)
df = df.dropna(subset=num_cols)
print(f"숫자 변환 후 결측 제거: {before_drop} → {len(df)} 행 사용")

# 2. 침수 취약도 지수 V 계산 ------------------------------------------------------
# 단순 예시: 각 그룹의 합으로 F, S, C 정의
F = df["FLOOD_AREA"] + df["RAIN_TOTAL"] + df["FLOOD_MONTH"]
S = df["RIV_GRD"] + df["RIV_DIS_GRD"] + df["RIV_DIS_MIN"]
C = df["DRAINAGE_GRD"] + df["PUMP_CNT"]

# C 가 0 이 되는 것을 방지하기 위해 작은 상수 추가
V = (F * S) / (C + 1e-6)

df["V_vulnerability"] = V

# V를 5구간(quintile)으로 나누어 0~4 등급 레이블 생성 (모델용)
# XGBoost 다중분류는 기본적으로 [0, 1, 2, 3, 4] 와 같이 0부터 시작하는 정수를 클래스로 사용합니다.
df["V_grade"] = pd.qcut(df["V_vulnerability"], q=5, labels=[0, 1, 2, 3, 4]).astype(int)

print("취약도 V 통계:")
print(df["V_vulnerability"].describe())
print("\n취약도 등급(모델용 0~4) 분포:")
print(df["V_grade"].value_counts(normalize=True).sort_index())

# 3. 특징(X), 타깃(y) 분리 ------------------------------------------------------
# - target: V_grade (1~5 등급, 분류 문제)
# - feature: 원본 설명변수들 (num_cols)

y = df["V_grade"]
X = df[num_cols]

print("\n원본 데이터 형태:", df.shape)
print("특징 행렬 X 형태:", X.shape)
print("타깃 벡터 y 형태:", y.shape)

# 4. 학습/검증용 데이터 분리 ------------------------------------------------------
# stratify=y : train/test 에서 1~5 등급 비율 유지
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("\nTrain X:", X_train.shape, " Test X:", X_test.shape)

# 5. 스케일링 --------------------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("스케일링 완료. 예시 첫 샘플(전처리 후) :", X_train_scaled[0][:5])


숫자 변환 후 결측 제거: 335377 → 333331 행 사용
취약도 V 통계:
count    3.333310e+05
mean     1.663129e+07
std      6.070997e+07
min      9.209117e+02
25%      3.250826e+05
50%      1.253160e+06
75%      4.864711e+06
max      7.048349e+08
Name: V_vulnerability, dtype: float64

취약도 등급(모델용 0~4) 분포:
V_grade
0    0.200002
1    0.199999
2    0.199999
3    0.199999
4    0.199999
Name: proportion, dtype: float64

원본 데이터 형태: (333331, 14)
특징 행렬 X 형태: (333331, 8)
타깃 벡터 y 형태: (333331,)

Train X: (266664, 8)  Test X: (66667, 8)
스케일링 완료. 예시 첫 샘플(전처리 후) : [-0.03998717 -0.81296005 -0.18919934  1.0671292  -0.20057782]


In [9]:
# 6. 기본 XGBoost 분류 모델 구성 ------------------------------------------------
# - 목적: 취약도 등급 1~5 를 예측하는 다중 클래스 분류
# - objective='multi:softmax' : 다중 클래스 분류, num_class=5

base_model = XGBClassifier(
    objective="multi:softmax",
    num_class=5,
    max_depth=5,
    learning_rate=0.1,
    n_estimators=300,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1,
)

# 모델 학습
base_model.fit(X_train_scaled, y_train)

# 검증 데이터 예측
y_pred_base = base_model.predict(X_test_scaled)

# 기본 모델 성능 확인 -----------------------------------------------------------
base_acc = accuracy_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base, average="weighted")

print("[Baseline XGBoost Classifier] 정확도:", round(base_acc, 4))
print("[Baseline XGBoost Classifier] F1(weighted):", round(base_f1, 4))
print("\n[Baseline] 상세 분류 리포트")
print(classification_report(y_test, y_pred_base))


[Baseline XGBoost Classifier] 정확도: 0.9992
[Baseline XGBoost Classifier] F1(weighted): 0.9992

[Baseline] 상세 분류 리포트
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13334
           1       1.00      1.00      1.00     13333
           2       1.00      1.00      1.00     13333
           3       1.00      1.00      1.00     13334
           4       1.00      1.00      1.00     13333

    accuracy                           1.00     66667
   macro avg       1.00      1.00      1.00     66667
weighted avg       1.00      1.00      1.00     66667



In [10]:
# 7. 하이퍼파라미터 탐색을 위한 설정 -------------------------------------------
# - RandomizedSearchCV 로 분류용 하이퍼파라미터 탐색
# - scoring: f1_weighted (클래스 불균형 고려)

from sklearn.model_selection import RandomizedSearchCV

# 탐색할 하이퍼파라미터 공간 정의
param_distributions = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.3, 1.0],
    "min_child_weight": [1, 3, 5],
}

# 분류용 베이스 모델 (1~5 등급 예측)
xgb_for_search = XGBClassifier(
    objective="multi:softmax",
    num_class=5,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
)

# RandomizedSearchCV: F1(weighted) 기준으로 최적 조합 탐색
random_search = RandomizedSearchCV(
    estimator=xgb_for_search,
    param_distributions=param_distributions,
    n_iter=30,
    scoring="f1_weighted",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42,
)

# 탐색 실행 (학습 + 검증 반복) --------------------------------------------------
random_search.fit(X_train_scaled, y_train)

print("\n[RandomizedSearchCV] 최적 하이퍼파라미터:")
print(random_search.best_params_)
print("[RandomizedSearchCV] CV 상 최고 F1(weighted):", round(random_search.best_score_, 4))


Fitting 3 folds for each of 30 candidates, totalling 90 fits

[RandomizedSearchCV] 최적 하이퍼파라미터:
{'subsample': 0.8, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 8, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 1.0}
[RandomizedSearchCV] CV 상 최고 F1(weighted): 0.9993


In [ ]:
# 8. 최적 하이퍼파라미터를 이용한 최종 모델 평가 -------------------------------
best_model = random_search.best_estimator_

# 테스트 데이터 예측 (1~5 등급)
y_pred_best = best_model.predict(X_test_scaled)

# 성능 평가 (정확도, F1, 분류 리포트)
best_acc = accuracy_score(y_test, y_pred_best)
best_f1 = f1_score(y_test, y_pred_best, average="weighted")

print("[Best XGBoost Classifier] 정확도:", round(best_acc, 4))
print("[Best XGBoost Classifier] F1(weighted):", round(best_f1, 4))
print("\n[Best] 상세 분류 리포트")
print(classification_report(y_test, y_pred_best))

# 베이스라인 대비 성능 비교 ------------------------------------------------------
print("\n[비교] Baseline vs Best (F1-weighted)")
print("Baseline F1:", round(base_f1, 4))
print("Best     F1:", round(best_f1, 4))


[Best XGBoost Classifier] 정확도: 0.9996
[Best XGBoost Classifier] F1(weighted): 0.9996

[Best] 상세 분류 리포트
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13334
           1       1.00      1.00      1.00     13333
           2       1.00      1.00      1.00     13333
           3       1.00      1.00      1.00     13334
           4       1.00      1.00      1.00     13333

    accuracy                           1.00     66667
   macro avg       1.00      1.00      1.00     66667
weighted avg       1.00      1.00      1.00     66667


[비교] Baseline vs Best (F1-weighted)
Baseline F1: 0.9992
Best     F1: 0.9996
